In [103]:
import sqlite3
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect("../market_data.db")

query = """
SELECT prices.*, symbols.sector, symbols.sub_sector
FROM prices
JOIN symbols ON prices.ticker = symbols.ticker
"""
df = pd.read_sql(query, conn)
df["date"] = pd.to_datetime(df["date"])
df.head()


Matplotlib is building the font cache; this may take a moment.


,ticker,date,open,high,low,close,volume,sector,sub_sector
0,NVDA,2019-01-02,3.231266,3.425181,3.216673,3.369282,508752000,Technology,Semiconductors
1,NVDA,2019-01-03,3.309177,3.343063,3.158299,3.165720,705552000,Technology,Semiconductors
2,NVDA,2019-01-04,3.238686,3.406630,3.208016,3.368540,585620000,Technology,Semiconductors
3,NVDA,2019-01-07,3.425676,3.583727,3.374476,3.546873,709160000,Technology,Semiconductors
4,NVDA,2019-01-08,3.628248,3.630475,3.386101,3.458572,786016000,Technology,Semiconductors


In [ ]:
#checking null values
df.isnull().sum()

ticker        0
date          0
open          0
high          0
low           0
close         0
volume        0
sector        0
sub_sector    0
dtype: int64

In [38]:
df.groupby('ticker')['close'].describe()

,count,mean,std,min,25%,50%,75%,max
ticker,,,,,,,,
ALHC,1382.0,13.174606,5.477578,4.470000,8.182500,12.730000,17.309999,27.270000
AMD,1944.0,126.338575,103.293060,17.049999,74.967503,101.750000,149.830006,630.630005
AXP,1944.0,187.409916,83.857767,63.396404,116.730659,157.389832,260.176376,381.781555
BIIB,1944.0,236.945643,57.017576,113.379997,199.587498,234.845001,279.252502,414.709991
BX,1944.0,92.026785,41.031359,22.046982,50.654799,93.386162,121.078278,187.206375
CRM,1944.0,213.743496,50.658650,121.971550,167.865398,209.657585,251.947853,362.581421
DDOG,1764.0,113.462225,45.293024,28.040001,87.707499,110.129997,132.615005,288.149994
JPM,1944.0,165.002019,78.318412,66.463875,105.668667,135.206680,213.854416,365.179993
KKR,1944.0,69.342531,38.619942,17.167793,37.060044,56.649326,100.751045,165.185257


In [30]:
df.groupby('sub_sector')['close'].describe()

,count,mean,std,min,25%,50%,75%,max
sub_sector,,,,,,,,
Asset Management,3888.0,80.684658,41.422273,17.167793,44.805235,77.095654,114.361834,187.206375
Banks,3888.0,107.075542,81.262022,18.450832,41.945239,84.295712,135.198807,365.179993
Credit Services,3888.0,213.150764,78.884395,63.396404,152.893208,201.873161,273.525673,384.140015
Drug Manufacturers,5832.0,264.301261,250.934551,56.134941,99.410364,191.065002,287.549995,1280.339966
Healthcare Plans,5270.0,234.040627,164.481245,4.470000,23.000000,256.500000,348.857513,596.105042
Semiconductors,5832.0,108.839680,141.329626,3.165720,40.061081,77.911606,123.198860,1213.373413
Software,5222.0,177.003616,70.187257,28.040001,127.052502,164.245461,230.435940,401.890015


In [43]:
df.groupby('ticker')[['sub_sector', 'date', 'close']].max()


,sub_sector,date,close
ticker,,,
ALHC,Healthcare Plans,2026-09-25,27.270000
AMD,Semiconductors,2026-09-25,630.630005
AXP,Credit Services,2026-09-25,381.781555
BIIB,Drug Manufacturers,2026-09-25,414.709991
BX,Asset Management,2026-09-25,187.206375
CRM,Software,2026-09-25,362.581421
DDOG,Software,2026-09-25,288.149994
JPM,Banks,2026-09-25,365.179993
KKR,Asset Management,2026-09-25,165.185257


In [60]:
# checking minimum values to flag anything that looks like $0 or near-zero - suggesting data error

df.groupby('ticker')[['date','open', 'close', 'high', 'low', 'volume']].min()

,date,open,close,high,low,volume
ticker,,,,,,
ALHC,2021-03-26,4.530000,4.470000,4.740000,4.460000,64700
AMD,2019-01-02,17.549999,17.049999,18.680000,16.940001,7956800
AXP,2019-01-02,66.687579,63.396404,69.335220,61.594517,665800
BIIB,2019-01-02,110.309998,113.379997,116.389999,110.040001,600
BX,2019-01-02,22.077151,22.046982,22.718273,21.503914,605500
CRM,2019-01-02,123.188306,121.971550,127.113363,113.130316,1238000
DDOG,2019-09-19,27.900000,28.040001,29.250000,27.549999,574800
JPM,2019-01-02,68.591632,66.463875,70.433375,64.681005,3220500
KKR,2019-01-02,17.200263,17.167793,17.874325,14.607540,771700


In [92]:
#To calculate how volatile my stock basket is we need to check the standard deviation of the stock returns on day to day basis

df_sorted = df.sort_values(['ticker', 'date'])
df_sorted['daily_return'] = df_sorted.groupby('ticker')['close'].pct_change()
volatility = df_sorted.groupby('ticker')['daily_return'].std().sort_values(ascending=False) * 100

# now to analyse this data, i want ticker name, volatility %age, sector name and sub_sector name
volatility = volatility.reset_index()
volatility.columns = ['ticker', 'volatility']
sector_column = df_sorted[['ticker', 'sector', 'sub_sector']].drop_duplicates()
volatility = volatility.merge(sector_column, on='ticker')
volatility = volatility.sort_values('volatility', ascending=False)
volatility

,ticker,volatility,sector,sub_sector
0,ALHC,4.093172,Healthcare,Healthcare Plans
1,SNOW,3.912621,Technology,Software
2,DDOG,3.791456,Technology,Software
3,AMD,3.527087,Technology,Semiconductors
4,MU,3.403021,Technology,Semiconductors
5,NVDA,3.178750,Technology,Semiconductors
6,BIIB,2.780632,Healthcare,Drug Manufacturers
7,MOH,2.611532,Healthcare,Healthcare Plans
8,KKR,2.485913,Financial Services,Asset Management
9,BX,2.454730,Financial Services,Asset Management


In [ ]:
#ALHC market cap in depth 

tickers_list = ['ALHC', 'UNH', 'NVDA']

for t in tickers_list:
    info = yf.Ticker(t).info
    market_cap = info.get('marketCap')
    print(f"{t}: ${market_cap:,}" if market_cap else f"{t}: not available")


ALHC: $1,647,035,136
UNH: $333,815,513,088
NVDA: $5,649,190,617,088


In [102]:
#ALHC volatility over time

alhc = df_sorted[df_sorted['ticker'] == 'ALHC'].copy()
alhc['year'] = alhc['date'].dt.year

alhc.groupby('year')['daily_return'].std()*100

year
2021    3.866416
2022    4.331350
2023    4.113053
2024    4.440756
2025    2.967486
2026    4.755599
Name: daily_return, dtype: float64

In [ ]:
#missing data, stock-split artifacts, outliers, distribution of returns, volatility by sector.

